In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")

air1 = pd.read_csv(DATA_DIR / "air_quality.csv")
weather = pd.read_csv(DATA_DIR / "weather_hourly.csv")
air2 = pd.read_csv(DATA_DIR / "multi_air_quality_hourly_20260304_224103.csv")

In [22]:
df = pd.concat([air1, air2], ignore_index=True)

In [23]:
#  BASIC CLEANING

# Standardize column names
df.columns = [col.strip().lower() for col in df.columns]

# Convert time column if it exists
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'], errors='coerce')

# Drop exact duplicates
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape after removing duplicates: (823008, 17)

Columns:
['city', 'state', 'zip', 'time', 'us_aqi', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'uv_index_clear_sky', 'uv_index', 'dust', 'aerosol_optical_depth', 'latitude', 'longitude']


In [24]:
#  CREATE TIME FEATURES

if 'time' in df.columns:
    df['date'] = df['time'].dt.date
    df['hour'] = df['time'].dt.hour
    df['day_name'] = df['time'].dt.day_name()
    df['month'] = df['time'].dt.month
    df['month_name'] = df['time'].dt.month_name()
    df['is_weekend'] = df['time'].dt.dayofweek >= 5
    df['day_type'] = np.where(df['is_weekend'], 'Weekend', 'Weekday')

    # Simple season mapping
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Spring'
        elif month in [6, 7, 8]:
            return 'Summer'
        else:
            return 'Fall'

    df['season'] = df['month'].apply(get_season)

print("Time features added.")
display(df.head())

Time features added.


,city,state,zip,time,us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,...,latitude,longitude,date,hour,day_name,month,month_name,is_weekend,day_type,season
0,Houston,TX,77002,2026-02-15 00:00:00-06:00,33.958336,10.4,10.0,242.0,32.0,4.1,...,NaN,NaN,2026-02-15,0,Sunday,2,February,True,Weekend,Winter
1,Houston,TX,77002,2026-02-15 01:00:00-06:00,33.142360,10.0,9.5,220.0,26.6,3.6,...,NaN,NaN,2026-02-15,1,Sunday,2,February,True,Weekend,Winter
2,Houston,TX,77002,2026-02-15 02:00:00-06:00,32.586807,9.6,9.1,196.0,17.3,2.8,...,NaN,NaN,2026-02-15,2,Sunday,2,February,True,Weekend,Winter
3,Houston,TX,77002,2026-02-15 03:00:00-06:00,32.256947,9.4,8.8,177.0,10.5,2.2,...,NaN,NaN,2026-02-15,3,Sunday,2,February,True,Weekend,Winter
4,Houston,TX,77002,2026-02-15 04:00:00-06:00,32.135420,8.8,7.9,165.0,8.8,2.2,...,NaN,NaN,2026-02-15,4,Sunday,2,February,True,Weekend,Winter


In [25]:
#  IDENTIFY KEY COLUMNS

# Candidate pollutant / numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove obvious metadata columns if present
metadata_like = ['zip', 'hour', 'month', 'is_weekend']
pollutant_cols = [col for col in numeric_cols if col not in metadata_like]

print("Numeric columns:")
print(numeric_cols)

print("\nCandidate pollutant / measurement columns:")
print(pollutant_cols)

Numeric columns:
['zip', 'us_aqi', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'uv_index_clear_sky', 'uv_index', 'dust', 'aerosol_optical_depth', 'latitude', 'longitude', 'hour', 'month']

Candidate pollutant / measurement columns:
['us_aqi', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'uv_index_clear_sky', 'uv_index', 'dust', 'aerosol_optical_depth', 'latitude', 'longitude']


In [26]:
#  DATA DICTIONARY TABLE

data_dict = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[col].dtype) for col in df.columns],
    'non_null_count': [df[col].notna().sum() for col in df.columns],
    'null_count': [df[col].isna().sum() for col in df.columns],
    'null_percent': [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    'n_unique': [df[col].nunique(dropna=True) for col in df.columns]
})

print("DATA DICTIONARY")
display(data_dict)

DATA DICTIONARY


,column,dtype,non_null_count,null_count,null_percent,n_unique
0,city,object,823008,0,0.00,51
1,state,object,823008,0,0.00,1
2,zip,int64,823008,0,0.00,532
3,time,"datetime64[ns, UTC-06:00]",823008,0,0.00,1512
4,us_aqi,float64,823008,0,0.00,11955
5,pm10,float64,823008,0,0.00,568
6,pm2_5,float64,823008,0,0.00,482
7,carbon_monoxide,float64,823008,0,0.00,1347
8,nitrogen_dioxide,float64,823008,0,0.00,825
9,sulphur_dioxide,float64,823008,0,0.00,239


In [27]:
#  DATA QUALITY SUMMARY

quality_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': [df[col].isna().sum() for col in df.columns],
    'missing_percent': [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    'duplicate_rows_total': [df.duplicated().sum()] * len(df.columns),
    'unique_values': [df[col].nunique(dropna=True) for col in df.columns]
})

print("DATA QUALITY SUMMARY")
display(quality_summary)

DATA QUALITY SUMMARY


,column,missing_count,missing_percent,duplicate_rows_total,unique_values
0,city,0,0.00,0,51
1,state,0,0.00,0,1
2,zip,0,0.00,0,532
3,time,0,0.00,0,1512
4,us_aqi,0,0.00,0,11955
5,pm10,0,0.00,0,568
6,pm2_5,0,0.00,0,482
7,carbon_monoxide,0,0.00,0,1347
8,nitrogen_dioxide,0,0.00,0,825
9,sulphur_dioxide,0,0.00,0,239


In [28]:
# DESCRIPTIVE STATISTICS TABLE

desc_stats = df[pollutant_cols].describe().T
desc_stats['median'] = df[pollutant_cols].median()
desc_stats['missing_count'] = df[pollutant_cols].isna().sum()
desc_stats['missing_percent'] = df[pollutant_cols].isna().mean() * 100
desc_stats['skewness'] = df[pollutant_cols].skew(numeric_only=True)
desc_stats = desc_stats[['count', 'missing_count', 'missing_percent', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'median', 'skewness']]

print("DESCRIPTIVE STATISTICS")
display(desc_stats)

DESCRIPTIVE STATISTICS


,count,missing_count,missing_percent,mean,std,min,25%,50%,75%,max,median,skewness
us_aqi,823008.0,0,0.000000,42.278461,14.169487,11.247681,32.274307,39.010410,50.097520,102.38540,39.010410,1.077370
pm10,823008.0,0,0.000000,10.777281,7.795289,0.000000,5.500000,8.800000,13.600000,77.50000,8.800000,1.791285
pm2_5,823008.0,0,0.000000,9.598348,7.208457,0.000000,4.900000,7.700000,11.900000,59.00000,7.700000,1.980160
carbon_monoxide,823008.0,0,0.000000,220.560498,96.077327,0.000000,171.000000,197.000000,239.000000,2740.00000,197.000000,5.250348
nitrogen_dioxide,823008.0,0,0.000000,14.445646,14.999863,0.000000,4.400000,9.100000,18.600000,104.40000,9.100000,1.972411
sulphur_dioxide,823008.0,0,0.000000,2.885840,2.275503,0.000000,1.400000,2.400000,3.700000,28.00000,2.400000,2.449241
ozone,823008.0,0,0.000000,61.884104,24.873387,0.000000,46.000000,63.000000,78.000000,142.00000,63.000000,-0.181866
uv_index_clear_sky,823008.0,0,0.000000,1.025937,1.695230,0.000000,0.000000,0.000000,1.600000,8.65000,0.000000,1.624110
uv_index,823008.0,0,0.000000,0.890338,1.530761,0.000000,0.000000,0.000000,1.250000,8.65000,0.000000,1.802234
dust,823008.0,0,0.000000,1.137996,5.284980,0.000000,0.000000,0.000000,1.000000,139.00000,0.000000,9.883602


In [29]:
#  MIN / MAX / AVG TABLE

min_max_avg = pd.DataFrame({
    'pollutant': pollutant_cols,
    'min': [df[col].min() for col in pollutant_cols],
    'avg': [df[col].mean() for col in pollutant_cols],
    'max': [df[col].max() for col in pollutant_cols]
})

print("MIN / AVG / MAX TABLE")
display(min_max_avg)

MIN / AVG / MAX TABLE


,pollutant,min,avg,max
0,us_aqi,11.247681,42.278461,102.38540
1,pm10,0.000000,10.777281,77.50000
2,pm2_5,0.000000,9.598348,59.00000
3,carbon_monoxide,0.000000,220.560498,2740.00000
4,nitrogen_dioxide,0.000000,14.445646,104.40000
5,sulphur_dioxide,0.000000,2.885840,28.00000
6,ozone,0.000000,61.884104,142.00000
7,uv_index_clear_sky,0.000000,1.025937,8.65000
8,uv_index,0.000000,0.890338,8.65000
9,dust,0.000000,1.137996,139.00000


In [30]:
#  AQI CATEGORY TABLE

if 'us_aqi' in df.columns:
    def aqi_category(x):
        if pd.isna(x):
            return np.nan
        elif x <= 50:
            return 'Good'
        elif x <= 100:
            return 'Moderate'
        elif x <= 150:
            return 'Unhealthy for Sensitive Groups'
        elif x <= 200:
            return 'Unhealthy'
        elif x <= 300:
            return 'Very Unhealthy'
        else:
            return 'Hazardous'

    df['aqi_category'] = df['us_aqi'].apply(aqi_category)

    aqi_category_table = df['aqi_category'].value_counts(dropna=False).reset_index()
    aqi_category_table.columns = ['aqi_category', 'count']
    aqi_category_table['percent'] = round(aqi_category_table['count'] / len(df) * 100, 2)

    print("AQI CATEGORY TABLE")
    display(aqi_category_table)

AQI CATEGORY TABLE


,aqi_category,count,percent
0,Good,615997,74.85
1,Moderate,206554,25.10
2,Unhealthy for Sensitive Groups,457,0.06


In [31]:
# ------------------------------------------------------------
# 10. HOURLY SUMMARY TABLE
# ------------------------------------------------------------

if 'hour' in df.columns:
    hourly_summary = df.groupby('hour')[pollutant_cols].agg(['mean', 'min', 'max']).round(3)
    print("HOURLY SUMMARY TABLE")
    display(hourly_summary)

HOURLY SUMMARY TABLE


us_aqi                     pm10              pm2_5             \
        mean     min      max    mean  min   max    mean  min   max   
hour                                                                  
0     42.921  15.480  101.875  12.187  0.3  56.8  11.012  0.3  56.1   
1     42.308  15.035  102.385  11.708  0.3  53.9  10.561  0.3  52.7   
2     41.844  15.139  101.490  11.314  0.3  55.5  10.184  0.3  49.6   
3     41.478  14.201  101.896  10.984  0.1  56.0   9.884  0.1  46.3   
4     41.202  14.031  101.906  10.718  0.1  56.6   9.654  0.1  47.7   
5     41.033  12.697  101.469  10.516  0.1  57.3   9.482  0.1  48.0   
6     40.926  11.827  100.854  10.206  0.0  63.8   9.186  0.0  41.1   
7     40.807  11.596  100.208  10.752  0.1  63.1   9.762  0.1  41.6   
8     40.627  11.422   99.486  11.753  0.2  69.6  10.765  0.2  44.9   
9     40.409  11.248   98.661  11.704  0.3  76.1  10.739  0.3  42.8   
10    40.227  11.306   97.863  10.732  0.2  71.7   9.663  0.2  39.5   
11    40.177  12.001   98.316   9.902  0.1  68.0   8.702  0.1  37.8   
12    40.307  12.581   99.326   9.352  0.2  70.3   8.080  0.2  37.6   
13    40.618  13.741   99.894   8.996  0.2  77.1   7.677  0.2  37.0   
14    41.111  15.422  100.292   8.784  0.1  77.5   7.439  0.1  36.2   
15    41.814  17.344  100.542   8.653  0.0  72.4   7.295  0.0  35.2   
16    42.742  17.188  100.635   8.599  0.0  70.3   7.235  0.0  32.9   
17    43.819  17.625  100.646   8.732  0.1  74.0   7.402  0.1  29.3   
18    44.931  18.321  100.240  10.507  0.0  75.4   9.204  0.0  57.5   
19    45.708  18.089   99.770  11.896  0.1  72.6  10.653  0.1  59.0   
20    45.907  18.089   99.078  12.590  0.2  67.1  11.363  0.2  58.4   
21    45.505  18.147   98.573  12.789  0.3  58.9  11.568  0.2  56.7   
22    44.612  17.899   99.326  12.742  0.2  57.3  11.521  0.2  55.6   
23    43.652  17.222  100.531  12.539  0.2  54.9  11.328  0.2  54.2   

     carbon_monoxide  ...   dust aerosol_optical_depth             latitude  \
                mean  ...    max                  mean   min   max     mean   
hour                  ...                                                     
0            237.366  ...   89.0                 0.123  0.01  0.66   31.073   
1            225.289  ...   84.0                 0.123  0.01  0.62   31.073   
2            213.211  ...   88.0                 0.124  0.01  0.58   31.073   
3            203.774  ...   91.0                 0.125  0.01  0.59   31.073   
4            196.011  ...   93.0                 0.127  0.01  0.71   31.073   
5            190.889  ...   93.0                 0.129  0.01  0.79   31.073   
6            193.967  ...  100.0                 0.130  0.01  0.80   31.073   
7            214.593  ...  100.0                 0.131  0.01  0.76   31.073   
8            243.416  ...  108.0                 0.132  0.01  0.66   31.073   
9            257.899  ...  132.0                 0.131  0.01  0.71   31.073   
10           243.875  ...  136.0                 0.128  0.01  0.61   31.073   
11           215.501  ...  130.0                 0.125  0.02  0.55   31.073   
12           192.795  ...  125.0                 0.123  0.02  0.55   31.073   
13           181.772  ...  128.0                 0.121  0.02  0.52   31.073   
14           176.404  ...  131.0                 0.120  0.02  0.53   31.073   
15           178.560  ...  131.0                 0.118  0.01  0.53   31.073   
16           191.987  ...  128.0                 0.117  0.02  0.54   31.073   
17           212.940  ...  139.0                 0.115  0.02  0.60   31.073   
18           232.010  ...  131.0                 0.134  0.01  0.94   31.073   
19           247.991  ...  121.0                 0.131  0.01  0.88   31.073   
20           262.099  ...  107.0                 0.129  0.01  0.76   31.073   
21           268.556  ...  100.0                 0.127  0.01  0.66   31.073   
22           262.919  ...  100.0                 0.127  0.01  0.63   31.073   
23           249.627  ...   95.0  

In [32]:
# ------------------------------------------------------------
# 11. WEEKDAY VS WEEKEND SUMMARY TABLE
# ------------------------------------------------------------

if 'day_type' in df.columns:
    weekday_weekend_summary = df.groupby('day_type')[pollutant_cols].agg(['mean', 'min', 'max', 'std']).round(3)
    print("WEEKDAY VS WEEKEND SUMMARY")
    display(weekday_weekend_summary)

WEEKDAY VS WEEKEND SUMMARY


us_aqi                             pm10                     pm2_5  \
            mean     min      max     std    mean  min   max    std    mean   
day_type                                                                      
Weekday   43.202  11.248  102.385  14.034  11.490  0.0  77.5  7.853  10.184   
Weekend   39.993  11.248   94.558  14.244   9.012  0.0  51.6  7.358   8.149   

               ... aerosol_optical_depth        latitude                  \
          min  ...                   max    std     mean     min     max   
day_type       ...                                                         
Weekday   0.0  ...                  0.94  0.094   31.073  25.951  35.375   
Weekend   0.0  ...                  0.71  0.083   31.073  25.951  35.375   

                longitude                          
            std      mean      min     max    std  
day_type                                           
Weekday   1.829   -97.724 -106.607 -94.106  2.582  
Weekend   1.829   -97.724 -106.607 -94.106  2.582  

[2 rows x 52 columns]

In [33]:
# ------------------------------------------------------------
# 14. ZIP CODE / LOCATION SUMMARY TABLE
# ------------------------------------------------------------

if 'zip' in df.columns:
    zip_summary = df.groupby('zip')[pollutant_cols].mean().round(3)

    if 'us_aqi' in df.columns:
        zip_summary = zip_summary.sort_values(by='us_aqi', ascending=False)

    print("ZIP CODE SUMMARY")
    display(zip_summary.head(20))

ZIP CODE SUMMARY


,us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,uv_index_clear_sky,uv_index,dust,aerosol_optical_depth,latitude,longitude
zip,,,,,,,,,,,,,
79920,49.561,19.264,12.371,436.954,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.823,-106.460
79911,49.561,19.264,12.371,260.012,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.936,-106.536
79908,49.561,19.264,12.371,387.589,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.830,-106.378
79906,49.561,19.264,12.371,387.589,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.809,-106.420
79934,49.561,19.264,12.371,204.590,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.952,-106.434
79930,49.561,19.264,12.371,436.954,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.811,-106.468
79904,49.561,19.264,12.371,260.012,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.850,-106.454
79924,49.561,19.264,12.371,229.525,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.902,-106.414
79922,49.561,19.264,12.371,369.878,23.716,5.773,56.864,1.116,1.034,13.548,0.067,31.811,-106.558


In [34]:
# ------------------------------------------------------------
# 15. TOP 10 ZIP CODES BY AQI
# ------------------------------------------------------------

if 'zip' in df.columns and 'us_aqi' in df.columns:
    top_zip_aqi = (
        df.groupby('zip')
        .agg(avg_aqi=('us_aqi', 'mean'),
             max_aqi=('us_aqi', 'max'),
             min_aqi=('us_aqi', 'min'),
             count=('us_aqi', 'count'))
        .sort_values(by='avg_aqi', ascending=False)
        .head(10)
        .round(3)
    )

    print("TOP 10 ZIP CODES BY AVG AQI")
    display(top_zip_aqi)

TOP 10 ZIP CODES BY AVG AQI


,avg_aqi,max_aqi,min_aqi,count
zip,,,,
79920,49.561,82.606,19.132,1512
79911,49.561,82.606,19.132,1512
79908,49.561,82.606,19.132,1512
79906,49.561,82.606,19.132,1512
79934,49.561,82.606,19.132,1512
79930,49.561,82.606,19.132,1512
79904,49.561,82.606,19.132,1512
79924,49.561,82.606,19.132,1512
79922,49.561,82.606,19.132,1512


In [35]:
# ------------------------------------------------------------
# 19. POLLUTANT DOMINANCE TABLE BY ZIP
# ------------------------------------------------------------

if 'zip' in df.columns:
    zip_means = df.groupby('zip')[pollutant_cols].mean()

    # Remove us_aqi from dominance competition if present
    pollutant_only_for_dominance = [col for col in pollutant_cols if col != 'us_aqi']

    dominant_pollutant_by_zip = pd.DataFrame({
        'zip': zip_means.index,
        'dominant_pollutant': zip_means[pollutant_only_for_dominance].idxmax(axis=1).values,
        'dominant_value': zip_means[pollutant_only_for_dominance].max(axis=1).values
    }).sort_values(by='dominant_value', ascending=False)

    print("DOMINANT POLLUTANT BY ZIP")
    display(dominant_pollutant_by_zip.head(20))

DOMINANT POLLUTANT BY ZIP


,zip,dominant_pollutant,dominant_value
519,79915,carbon_monoxide,578.678571
510,79902,carbon_monoxide,436.953704
520,79920,carbon_monoxide,436.953704
526,79930,carbon_monoxide,436.953704
518,79912,carbon_monoxide,436.953704
509,79901,carbon_monoxide,436.953704
514,79906,carbon_monoxide,387.588624
511,79903,carbon_monoxide,387.588624
513,79905,carbon_monoxide,387.588624
516,79908,carbon_monoxide,387.588624


In [36]:
# ------------------------------------------------------------
# 22. SIMPLE OUTLIER TABLE USING IQR
# ------------------------------------------------------------

outlier_results = []

for col in pollutant_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = df[(df[col] < lower) | (df[col] > upper)][col].count()

    outlier_results.append({
        'pollutant': col,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': outlier_count,
        'outlier_percent': round(outlier_count / len(df) * 100, 2)
    })

outlier_table = pd.DataFrame(outlier_results).sort_values(by='outlier_percent', ascending=False)

print("OUTLIER TABLE")
display(outlier_table)

OUTLIER TABLE


,pollutant,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percent
8,uv_index,0.000000,1.250000,1.250000,-1.875000,3.125000,98635,11.98
7,uv_index_clear_sky,0.000000,1.600000,1.600000,-2.400000,4.000000,73947,8.98
4,nitrogen_dioxide,4.400000,18.600000,14.200000,-16.900000,39.900000,63449,7.71
9,dust,0.000000,1.000000,1.000000,-1.500000,2.500000,56345,6.85
3,carbon_monoxide,171.000000,239.000000,68.000000,69.000000,341.000000,52572,6.39
2,pm2_5,4.900000,11.900000,7.000000,-5.600000,22.400000,51006,6.20
1,pm10,5.500000,13.600000,8.100000,-6.650000,25.750000,44220,5.37
5,sulphur_dioxide,1.400000,3.700000,2.300000,-2.050000,7.150000,41258,5.01
10,aerosol_optical_depth,0.060000,0.160000,0.100000,-0.090000,0.310000,38149,4.64
12,longitude,-98.389933,-95.659630,2.730303,-102.485386,-91.564176,36288,4.41


In [37]:
# ------------------------------------------------------------
# 24. SUMMARY TABLE FOR PRESENTATION
# ------------------------------------------------------------

summary_rows = []

summary_rows.append(['Total rows', len(df)])
summary_rows.append(['Total columns', len(df.columns)])

if 'time' in df.columns:
    summary_rows.append(['Start time', df['time'].min()])
    summary_rows.append(['End time', df['time'].max()])

if 'zip' in df.columns:
    summary_rows.append(['Unique zip codes', df['zip'].nunique()])

if 'city' in df.columns:
    summary_rows.append(['Unique cities', df['city'].nunique()])

if 'us_aqi' in df.columns:
    summary_rows.append(['Average AQI', round(df['us_aqi'].mean(), 3)])
    summary_rows.append(['Minimum AQI', round(df['us_aqi'].min(), 3)])
    summary_rows.append(['Maximum AQI', round(df['us_aqi'].max(), 3)])

project_summary = pd.DataFrame(summary_rows, columns=['metric', 'value'])

print("PROJECT SUMMARY TABLE")
display(project_summary)

PROJECT SUMMARY TABLE


,metric,value
0,Total rows,823008
1,Total columns,26
2,Start time,2026-01-01 00:00:00-06:00
3,End time,2026-03-04 23:00:00-06:00
4,Unique zip codes,532
5,Unique cities,51
6,Average AQI,42.278
7,Minimum AQI,11.248
8,Maximum AQI,102.385


In [38]:
# ------------------------------------------------------------
# 25. EXPORT ALL TABLES AS DOWNLOADABLE FILES
# ------------------------------------------------------------

import os

print("Generated CSV files:")
for file in os.listdir():
    if file.startswith("table_") and file.endswith(".csv"):
        print(file)

Generated CSV files:
